# Denoising 4: Denoising Autoencoder on the Features → GDBN

Notebooks 14–16 clean the **returns** (remove the shared part). This one cleans **each stock's whole feature vector** instead: the classic **denoising autoencoder** from the vault notes ("Notes - Financial Volatility Prediction Model Based On Denoising Autoencoder..." and the autoencoder background in "Notes - Strong denoising of financial time-series").

```
one stock-day: 25 features (z-scored)
        |   add noise on purpose (Gaussian noise + randomly blank out 20% of the values)
        v
   [25] -> [32] -> [ 8 ]  <- the waist  -> [32] -> [25]
        encoder              (summary)        decoder
        |
        v
must rebuild the ORIGINAL (clean) 25 features  ->  cleaned excess return, sentiment, attention  ->  GDBN
```

- **Why this removes noise:** the waist (8 numbers) is too small to store all 25 features, and the network is graded on rebuilding the **clean** version from a **noisy** one. So it keeps the patterns the features share and drops one-off wiggles.
- **Missing values:** sentiment is missing on no-news days. The network is only graded on values that exist, so its output on a no-news day is its best guess from the other features (a learned fill-in).
- **What the GDBN gets:** the cleaned versions of its three stock nodes: **excess return, sentiment, attention**. ETFs stay as they are.

| | Setting |
|---|---|
| Rows | every stock-day with data (about 600 stocks a day), z-scored with the fitting years' mean and standard deviation, clipped at ±5 |
| Network | 25 → 32 → **8** → 32 → 25 (ReLU) |
| Corruption | Gaussian noise (std 0.5) + 20% of values set to 0 |
| Training | each year on the previous 5 years, batches of 2,048 rows; the last year of the window for early stopping (rebuild error) |

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import time
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pit_common as pc
import denoise_common as dc

D = pc.load()                        # returns and universe for the GDBN (everything after 2022 blanked)
cube = dc.load_cube()                # the feature table (features_daily.csv.gz) as arrays
GDBN_MASK = dc.gdbn_mask(D, cube)    # the GDBN's 52-stock universe, for R^2 on its own stocks


def frame(arr, rows, cols=None):
    return pd.DataFrame(arr, index=cube.days[rows], columns=cube.perm if cols is None else cols)

## 1. Train each year and clean the features

In [ ]:
import torch
import torch.nn as nn
torch.set_num_threads(8)

NOISE, DROP, WAIST, HIDDEN, BATCH, LR, MAX_EPOCHS, PATIENCE, VAL_DAYS = 0.5, 0.2, 8, 32, 2048, 1e-3, 15, 2, 252
NODES = ["exret_1d", "sentiment", "attention"]                     # the GDBN's stock nodes
C = len(dc.FEATS)


class DAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(C, HIDDEN), nn.ReLU(), nn.Linear(HIDDEN, WAIST))
        self.dec = nn.Sequential(nn.ReLU(), nn.Linear(WAIST, HIDDEN), nn.ReLU(), nn.Linear(HIDDEN, C))

    def forward(self, x):
        return self.dec(self.enc(x))


def rows_of(days_, mu, sd):
    x = cube.X[days_][cube.AVAIL[days_]]                            # stock-days with data
    z = (x - mu) / sd
    obs = np.isfinite(z)
    return torch.from_numpy(np.clip(np.nan_to_num(z), -5, 5).astype(np.float32)), torch.from_numpy(obs.astype(np.float32))


def masked_mse(out, x, obs):
    return float((((out - x) ** 2) * obs).sum() / obs.sum())


by_year, rows = {}, []
t0 = time.time()
for Y in dc.YEARS:
    fd, ad = dc.fit_days(cube, Y), dc.apply_days(cube, Y)
    sl = cube.X[fd][cube.AVAIL[fd]]
    mu, sd = np.nanmean(sl, axis=0), np.nanstd(sl, axis=0) + 1e-6
    xtr, otr = rows_of(fd[:-VAL_DAYS], mu, sd)
    xva, ova = rows_of(fd[-VAL_DAYS:], mu, sd)
    torch.manual_seed(Y)
    model = DAE()
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    best, state, bad, passes = np.inf, None, 0, 0
    for epoch in range(MAX_EPOCHS):
        model.train()
        for idx in torch.randperm(len(xtr)).split(BATCH):
            xb, ob = xtr[idx], otr[idx]
            noisy = (xb + NOISE * torch.randn_like(xb) * ob) * (torch.rand_like(xb) > DROP)
            loss = (((model(noisy) - xb) ** 2) * ob).sum() / ob.sum()
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            score = masked_mse(model(xva), xva, ova)
        passes += 1
        if score < best:
            best, state, bad = score, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(state)
    model.eval()

    clean = np.full((len(ad), len(cube.perm), C), np.nan, dtype=np.float32)
    xa, oa = rows_of(ad, mu, sd)
    with torch.no_grad():
        out = model(xa).numpy() * sd + mu                           # back to the original units
    clean[cube.AVAIL[ad]] = out
    j = {f: dc.FEATS.index(f) for f in NODES}
    by_year[Y] = SimpleNamespace(RET=frame(clean[:, :, j["exret_1d"]], ad), SENT=frame(clean[:, :, j["sentiment"]], ad),
                                 ATTN=frame(clean[:, :, j["attention"]], ad), FACT=None)

    yd = cube.days[ad].year == Y                                    # out-of-sample rebuild quality, by feature
    mem = cube.MEM[ad[yd]]
    x_all, c_all = cube.X[ad[yd]][mem], clean[yd][mem]              # member stock-days of year Y
    row = {"year": Y, "passes": passes, "validation error (z units)": best}
    for f in NODES + ["(all 25 features)"]:
        cols = [j[f]] if f in j else list(range(C))
        x_, c_ = x_all[:, cols], c_all[:, cols]
        ok = np.isfinite(x_) & np.isfinite(c_)
        row[f"R2 {f}"] = 1 - (((c_ - x_) ** 2)[ok]).sum() / (((x_ - mu[cols]) ** 2)[ok]).sum()
    rows.append(row)
    print(f"  {Y}: {passes} passes, validation error {best:.3f} ({(time.time() - t0) / 60:.1f} min)")
r2_table = pd.DataFrame(rows).set_index("year")
r2_table.round(3)

**Reading it:** for each feature, **R²** = how much of its day-to-day variation the cleaned version keeps, on days not used for training. A high R² means the feature is mostly "pattern" the autoencoder can rebuild; a low R² means most of it was treated as noise and smoothed away. Daily excess return is expected to be low (it is mostly noise); attention should be high.

### Example: one stock's raw vs cleaned series

In [ ]:
aapl = list(cube.perm).index(14593)
Yx = 2019
V = by_year[Yx]
days_x = V.RET.loc[f"{Yx}-01-01":f"{Yx}-03-31"].index
rows_x = cube.days.get_indexer(days_x)
fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
for ax, f, cl in zip(axes, NODES, [V.RET, V.SENT, V.ATTN]):
    ax.plot(days_x, cube.X[rows_x, aapl, dc.FEATS.index(f)], ".", label="raw", alpha=0.6)
    ax.plot(days_x, cl.loc[days_x, 14593], "-", label="cleaned", lw=2)
    ax.set_title(f"Apple, {f}, Q1 {Yx}"); ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%b %d")); ax.tick_params(axis="x", labelsize=8)
plt.tight_layout(); plt.show()

## GDBN on the denoised data

Four versions of the GDBN, all on the same 52-stock point-in-time universe, walk-forward, λ rule (best rank IC on 2006–2020), and 2021–2022 validation:

| Version | Inputs (yesterday's values) |
|---|---|
| **raw daily** | excess return, sentiment, attention, ETFs (notebook 06) |
| **raw slow** | past-week and past-month averages of the same (notebook 10, the best so far) |
| **DAE daily** | the same, with the **denoised** series in place of the raw one: cleaned excess return, cleaned sentiment, cleaned attention (no factor nodes) |
| **DAE slow** | past-week and past-month averages of the denoised series |

The target is unchanged: each stock's next-day excess return. The raw versions are saved the first time and loaded in later notebooks.

**Columns:** `arrows/quarter` = network size at the chosen λ; `arrow stability` = how much consecutive quarters' networks overlap (1 = identical). Denser networks overlap more just by chance, so `arrows @ raw λ` and `stability @ raw λ` repeat both at the **same λ as the raw GDBN** of the same kind: the fair, like-for-like comparison. Then rank IC, Newey–West t, gross Sharpe (before costs), and breakeven bps (the cost level that would wipe out the profit) for each period.

In [ ]:
runs = dc.raw_gdbn(D)
runs["DAE daily"] = dc.walk_forward("dae_daily", D, dc.gdbn_features(by_year, "daily"))
runs["DAE slow"] = dc.walk_forward("dae_slow", D, dc.gdbn_features(by_year, "slow"))
table, scores = dc.evaluate(D, runs)
print(table.round(4).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for name, s in scores.items():
    ic = pc.rank_ic(s, pc.forward_excess(D, s, 1))
    axes[0].plot(ic.cumsum(), label=name, ls="--" if name.startswith("raw") else "-")
for name, res in runs.items():
    k, _ = pc.choose_lambda(D, res, 1)
    axes[1].plot([q["r"] for q in res], [(q["W"][:, :, k] != 0).sum() for q in res], label=name,
                 ls="--" if name.startswith("raw") else "-")
axes[0].axvline(pd.Timestamp("2021-01-01"), color="gray", lw=1); axes[0].set_title("Cumulative rank IC (rising = skill; line = validation starts)")
axes[1].set_title("Arrows in each quarter's network (at each version's chosen λ)")
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**Reading it:**
- **Did denoising help prediction?** Compare the DAE rows with the raw rows of the same kind (daily vs daily, slow vs slow): rank IC, gross Sharpe, breakeven, and especially **2021–2022**.
- **Did it clean up the connections?** Compare **stability @ raw λ** (same λ, so the same price per arrow): higher means the network keeps finding the same links from quarter to quarter, which is what real (not noise-driven) connections should do. The plain `arrow stability` column is inflated for versions that chose a smaller λ (bigger networks).
- Notebook 18 puts all four denoisers side by side.

In [ ]:
dc.save("dae", table, r2_table)